# 5G Network Logs: Data Cleaning and Preprocessing

This Google Colab notebook manually uploads the synthetic 5G log dataset, prepares it for
TF-IDF and Word2Vec experiments, and downloads the cleaned CSV to your computer.

**Output:** `cleaned_5g_network_logs.csv`

**How to use it:** Run every cell, choose `synthetic_5g_network_logs_50000.csv` when the
upload window opens, and allow your browser to download the cleaned CSV at the end.

Design choices:

- Preserve the original `log_text` for auditability.
- Preserve severity words because they are genuine log content.
- Create `clean_text` by removing timestamp and identifier variability.
- Preserve 5G terms, operational verbs and negation; do not remove stop words or stem words.
- Keep the supplied scenario/session-based train, validation and test splits.

In [ ]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
from google.colab import files

CLEAN_CSV = Path("/content/cleaned_5g_network_logs.csv")
RANDOM_STATE = 7_002_026

REQUIRED_COLUMNS = [
    "timestamp", "network_function", "severity", "log_text", "label",
    "anomaly_type", "scenario_id", "session_id", "template_id", "split",
]

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)

## 1. Load and validate the raw dataset

Select the raw CSV from your computer when Colab opens the upload window. The raw CSV remains
unchanged. All transformations are written to a separate cleaned CSV in Colab's temporary
`/content` folder.

In [ ]:
EXPECTED_RAW_NAME = "synthetic_5g_network_logs_50000.csv"
print(f"Upload {EXPECTED_RAW_NAME}")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No CSV file was uploaded.")
if len(uploaded) != 1:
    raise ValueError("Upload exactly one raw CSV file, then run this cell again.")

uploaded_name = next(iter(uploaded))
RAW_CSV = Path("/content") / uploaded_name
if RAW_CSV.name != EXPECTED_RAW_NAME:
    print(f"Warning: uploaded {RAW_CSV.name}; expected {EXPECTED_RAW_NAME}.")
if not RAW_CSV.exists():
    raise FileNotFoundError(f"Colab could not find the uploaded file at {RAW_CSV}.")

raw_df = pd.read_csv(RAW_CSV, dtype="string", keep_default_na=False)
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = raw_df[REQUIRED_COLUMNS].copy()
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns from {RAW_CSV.name}")
print(df.head(3).to_string(index=False))

In [ ]:
parsed_time = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
audit_before = {
    "rows": len(df),
    "empty_cells": int(df.eq("").sum().sum()),
    "invalid_timestamps": int(parsed_time.isna().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "duplicate_log_text": int(df["log_text"].duplicated().sum()),
    "labels": sorted(df["label"].unique().tolist()),
    "severities": sorted(df["severity"].unique().tolist()),
    "splits": sorted(df["split"].unique().tolist()),
}
print(pd.Series(audit_before, name="value").to_string())
print("\nClass by split:\n", pd.crosstab(df["split"], df["label"]))
print("\nSeverity by class:\n", pd.crosstab(df["severity"], df["label"]))

## 2. Clean and normalise the modelling text

Dynamic identifiers are replaced with semantic tokens such as `supi_id`, `session_id` and
`ip_address`. Numeric values become `num_value`. This prevents TF-IDF from learning thousands
of one-off identifiers and gives Word2Vec reusable contexts.

Severity tokens such as `info`, `warning`, `error` and `critical` are deliberately retained.

In [ ]:
LEADING_TIMESTAMP_RE = re.compile(
    r"^\s*\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}(?:\.\d+)?\s*"
)
REPLACEMENTS = [
    (re.compile(r"\bsupi-\d+\b", re.I), "supi_id"),
    (re.compile(r"\bguti-[0-9a-f]+\b", re.I), "guti_id"),
    (re.compile(r"\bses-\d+\b", re.I), "session_id"),
    (re.compile(r"\bgnb-\d+\b", re.I), "gnb_id"),
    (re.compile(r"\bcell-\d+\b", re.I), "cell_id"),
    (re.compile(r"\bnf-\d+\b", re.I), "nf_instance_id"),
    (re.compile(r"\bpol-\d+\b", re.I), "policy_id"),
    (re.compile(r"\bsec-\d+\b", re.I), "security_context_id"),
    (re.compile(r"\bueg-\d+\b", re.I), "ue_group_id"),
    (re.compile(r"\bsst-\d+-sd-[0-9a-f]+\b", re.I), "slice_id"),
    (re.compile(r"\b(?:10|172|192)\.\d{1,3}\.\d{1,3}\.\d{1,3}(?::\d+)?\b"), "ip_address"),
    (re.compile(r"\b\d+(?:\.\d+)?\b"), "num_value"),
]


def clean_log_text(value: str) -> str:
    """Create conservative, word-focused text without changing event meaning."""
    text = LEADING_TIMESTAMP_RE.sub("", str(value)).lower()
    for pattern, replacement in REPLACEMENTS:
        text = pattern.sub(replacement, text)
    text = re.sub(r"[^a-z0-9_\-\s]", " ", text)
    text = text.replace("-", "_")
    return re.sub(r"\s+", " ", text).strip()


# Trim text fields, remove only exact duplicate rows and retain the raw log message.
for column in REQUIRED_COLUMNS:
    df[column] = df[column].str.strip()
df = df.drop_duplicates().copy()

label_conflicts = df.groupby("log_text")["label"].nunique()
conflicting_messages = int(label_conflicts.gt(1).sum())
if conflicting_messages:
    raise ValueError(
        f"Found {conflicting_messages} identical messages with contradictory labels."
    )

df["clean_text"] = df["log_text"].map(clean_log_text)
df["target"] = df["label"].map({"normal": 0, "abnormal": 1}).astype("int8")
if df["clean_text"].eq("").any():
    raise ValueError("Cleaning produced one or more empty text records.")

# Chronological order is retained; the predefined split column remains unchanged.
df["_parsed_timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="raise")
df = df.sort_values("_parsed_timestamp", kind="stable").drop(columns="_parsed_timestamp")
df = df.reset_index(drop=True)

In [ ]:
before_after = pd.DataFrame({
    "raw_log_text": df["log_text"].head(8),
    "clean_text": df["clean_text"].head(8),
})
print(before_after.to_string(index=False))

## 3. Validate research splits and cleaned output

These checks stop the notebook if preprocessing changes the agreed study design or creates
train/test leakage through scenarios or sessions.

In [ ]:
EXPECTED_CLASS_COUNTS = {"normal": 37_500, "abnormal": 12_500}
EXPECTED_SPLIT_COUNTS = {"train": 35_000, "validation": 7_500, "test": 7_500}
EXPECTED_SEVERITY_COUNTS = {
    ("normal", "INFO"): 27_000,
    ("normal", "DEBUG"): 6_000,
    ("normal", "NOTICE"): 4_000,
    ("normal", "WARNING"): 500,
    ("abnormal", "INFO"): 500,
    ("abnormal", "WARNING"): 4_500,
    ("abnormal", "ERROR"): 6_500,
    ("abnormal", "CRITICAL"): 1_000,
}

class_counts = df["label"].value_counts().to_dict()
split_counts = df["split"].value_counts().to_dict()
severity_counts = df.groupby(["label", "severity"]).size().to_dict()
scenario_leakage = int(df.groupby("scenario_id")["split"].nunique().gt(1).sum())
session_leakage = int(df.groupby("session_id")["split"].nunique().gt(1).sum())

assert len(df) == 50_000
assert class_counts == EXPECTED_CLASS_COUNTS
assert split_counts == EXPECTED_SPLIT_COUNTS
assert severity_counts == EXPECTED_SEVERITY_COUNTS
assert scenario_leakage == 0
assert session_leakage == 0
assert not df["clean_text"].str.contains(r"\b(?:normal|abnormal)\b", case=False, regex=True).any()

token_lengths = df["clean_text"].str.split().str.len()
quality_summary = pd.Series({
    "rows_after_cleaning": len(df),
    "unique_raw_logs": df["log_text"].nunique(),
    "unique_clean_texts": df["clean_text"].nunique(),
    "mean_clean_tokens": round(float(token_lengths.mean()), 2),
    "minimum_clean_tokens": int(token_lengths.min()),
    "maximum_clean_tokens": int(token_lengths.max()),
    "severity_exceptions": int(
        ((df["label"] == "normal") & (df["severity"] == "WARNING")).sum()
        + ((df["label"] == "abnormal") & (df["severity"] == "INFO")).sum()
    ),
    "scenario_split_leakage": scenario_leakage,
    "session_split_leakage": session_leakage,
}, name="value")
print(quality_summary.to_string())

## 4. Export the cleaned dataset

The second notebook reads this CSV only. It does not read the raw generated dataset. After
the file is written, Colab starts a browser download automatically.

In [ ]:
OUTPUT_COLUMNS = [
    "timestamp", "network_function", "severity", "log_text", "clean_text",
    "label", "target", "anomaly_type", "scenario_id", "session_id",
    "template_id", "split",
]
df[OUTPUT_COLUMNS].to_csv(CLEAN_CSV, index=False, encoding="utf-8")

sha256 = hashlib.sha256(CLEAN_CSV.read_bytes()).hexdigest()
print(f"Created {CLEAN_CSV.resolve()}")
print(f"Rows: {len(df):,}")
print(f"SHA-256: {sha256}")
print("Starting download of the cleaned CSV...")
files.download(str(CLEAN_CSV))

## Output contract

The cleaned CSV contains both `log_text` and `clean_text`. Use `clean_text` as the only text
input in the six experiments, and use `target` as the binary outcome. Metadata columns are
retained for evaluation and leakage checks, not as model features.